In [ ]:
import zipfile
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils import class_weight

EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

def extract_dataset(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('ravdess_data')

    for root, dirs, files in os.walk('ravdess_data'):
        if any(d.startswith('Actor_') for d in dirs):
            print(f" Found audio directory at: {root}")
            return root
    return 'ravdess_data'

def create_emotion_model():
    model = keras.Sequential([
        layers.Input(shape=(40, 100, 3)),
        layers.Conv2D(64, (3, 3), activation='elu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(128, (3, 3), activation='elu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation='elu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(len(EMOTIONS), activation='softmax')
    ])

    optimizer = keras.optimizers.Adam(learning_rate=0.0001)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
    )
    return model

def extract_features(audio_data, sr=16000):
    try:
        audio_data, _ = librosa.effects.trim(audio_data)
        audio_data = librosa.util.fix_length(audio_data, size=sr * 3)

        mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=40)

        delta_mfccs = librosa.feature.delta(mfccs)
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)

        features = np.stack([mfccs, delta_mfccs, delta2_mfccs], axis=-1)
        features = (features - np.mean(features)) / (np.std(features) + 1e-6)

        target_length = 100
        if features.shape[1] < target_length:
            features = np.pad(features, ((0, 0), (0, target_length - features.shape[1]), (0, 0)), mode='constant')
        else:
            features = features[:, :target_length, :]

        return features
    except Exception as e:
        print(f"Feature error: {e}")
        return None

def train_emotion_model(train_data_path, model_save_path='emotion_model.keras', epochs=100, batch_size=32):
    X_all, y_all = [], []

    print(f"Loading files from {train_data_path}...")

    for root, _, files in os.walk(train_data_path):
        for file in files:
            if file.endswith('.wav'):
                parts = file.split('-')
                if len(parts) < 3: continue

                emotion_num = int(parts[2])
                if emotion_num <= 2: label = 0
                else: label = emotion_num - 2

                try:
                    audio_data, sr = librosa.load(os.path.join(root, file), sr=16000)
                    features = extract_features(audio_data, sr)
                    if features is not None:
                        X_all.append(features)
                        y_all.append(label)
                except Exception: continue

    X_all = np.array(X_all)
    y_all = np.array(y_all)

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_all), y=y_all)
    class_weights = dict(enumerate(weights))

    indices = np.arange(len(X_all))
    np.random.shuffle(indices)
    X_all = X_all[indices]
    y_all = y_all[indices]

    y_all_cat = keras.utils.to_categorical(y_all, len(EMOTIONS))

    model = create_emotion_model()

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
    ]

    print(f"Training on {len(X_all)} samples with 3-channel features...")
    model.fit(
        X_all, y_all_cat,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        shuffle=True,
        class_weight=class_weights,
        callbacks=callbacks
    )

    print("\n--- Final Performance Report ---")
    predictions = model.predict(X_all)
    y_pred_labels = np.argmax(predictions, axis=1)
    print(classification_report(y_all, y_pred_labels, target_names=EMOTIONS))

    model.save(model_save_path)
    return model

if __name__ == "__main__":
    ZIP_FILE_NAME = '/content/Ravdess-20260214T111346Z-1-001.zip'

    if os.path.exists(ZIP_FILE_NAME):
        DATA_PATH = extract_dataset(ZIP_FILE_NAME)
        train_emotion_model(DATA_PATH)
    else:
        print(f"Error: {ZIP_FILE_NAME} not found.")

Extracting /content/Ravdess-20260214T111346Z-1-001.zip...
✓ Found audio directory at: ravdess_data/Ravdess/audio_speech_actors_01-24
Loading files from ravdess_data/Ravdess/audio_speech_actors_01-24...
Training on 1440 samples with 3-channel features...
Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.2266 - loss: 2.8872 - precision: 0.2489 - recall: 0.1616 - val_accuracy: 0.2188 - val_loss: 2.0465 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3091 - loss: 2.3563 - precision: 0.3450 - recall: 0.2113 - val_accuracy: 0.2118 - val_loss: 2.5559 - val_precision: 0.3090 - val_recall: 0.1910 - learning_rate: 1.0000e-04
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.3993 - loss: 1.8691 - precision: 0.4461 - recall: 0.2904 - val_accuracy: 0.2222 - val_loss: 2.9089 - val_precision: 0.2574 - val_recall: 0.2118 - learning_rate: 1.0000e-04
Epoch 4/100
36/36 ━━━━━━

In [ ]:
import zipfile
import os
import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report
from sklearn.utils import class_weight

EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

def extract_dataset(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('aggregated_data')
    return 'aggregated_data'

def get_label_from_file(file_name, root_path):
    fn = file_name.lower()
    path = root_path.lower()

    if 'tess' in path or 'oaf_' in fn or 'yaf_' in fn:
        if 'neutral' in fn: return 0
        if 'happy' in fn: return 1
        if 'sad' in fn: return 2
        if 'angry' in fn: return 3
        if 'fear' in fn: return 4
        if 'disgust' in fn: return 5
        if 'pleasant_surprised' in fn or 'surprise' in fn: return 6

    if fn.startswith('03-') or '-' in fn:
        parts = fn.split('-')
        if len(parts) >= 3:
            num = int(parts[2])
            if num <= 2: return 0
            return num - 2

    if '_ang_' in fn: return 3
    if '_hap_' in fn: return 1
    if '_sad_' in fn: return 2
    if '_fea_' in fn: return 4
    if '_dis_' in fn: return 5
    if '_neu_' in fn: return 0

    if 'savee' in path or fn.startswith('dc_') or fn.startswith('jk_'):
        code = fn.split('_')[1][0] if '_' in fn else fn[0]
        mapping = {'n':0, 'h':1, 's':2, 'a':3, 'f':4, 'd':5, 'u':6}
        return mapping.get(code, None)

    return None

def create_emotion_model():
    model = keras.Sequential([
        layers.Input(shape=(40, 100, 3)),
        layers.Conv2D(64, (3, 3), activation='elu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(128, (3, 3), activation='elu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        layers.Conv2D(256, (3, 3), activation='elu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),

        layers.Flatten(),
        layers.Dense(512, activation='elu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(len(EMOTIONS), activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def extract_features(audio_data, sr=16000):
    try:
        audio_data, _ = librosa.effects.trim(audio_data)
        audio_data = librosa.util.fix_length(audio_data, size=sr * 3)
        mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=40)
        delta_mfccs = librosa.feature.delta(mfccs)
        delta2_mfccs = librosa.feature.delta(mfccs, order=2)
        features = np.stack([mfccs, delta_mfccs, delta2_mfccs], axis=-1)
        features = (features - np.mean(features)) / (np.std(features) + 1e-6)

        target_length = 100
        if features.shape[1] < target_length:
            features = np.pad(features, ((0, 0), (0, target_length - features.shape[1]), (0, 0)), mode='constant')
        else:
            features = features[:, :target_length, :]
        return features
    except Exception:
        return None

def train_emotion_model(train_data_path, model_save_path='emotion_model.keras', epochs=100):
    X_all, y_all = [], []
    print("Loading datasets (Crema, Ravdess, Savee, Tess)...")

    for root, _, files in os.walk(train_data_path):
        for file in files:
            if file.endswith('.wav'):
                label = get_label_from_file(file, root)
                if label is not None:
                    try:
                        audio_data, sr = librosa.load(os.path.join(root, file), sr=16000)
                        features = extract_features(audio_data, sr)
                        if features is not None:
                            X_all.append(features)
                            y_all.append(label)
                    except: continue

    X_all, y_all = np.array(X_all), np.array(y_all)

    weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_all), y=y_all)
    class_weights = dict(enumerate(weights))

    indices = np.arange(len(X_all))
    np.random.shuffle(indices)
    X_all, y_all = X_all[indices], y_all[indices]
    y_all_cat = keras.utils.to_categorical(y_all, len(EMOTIONS))

    model = create_emotion_model()
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1)
    ]

    print(f"Total samples across all datasets: {len(X_all)}")
    model.fit(X_all, y_all_cat, epochs=epochs, batch_size=32, validation_split=0.15,
              shuffle=True, class_weight=class_weights, callbacks=callbacks)

    print("\n--- Consolidated Performance Report ---")
    predictions = model.predict(X_all)
    print(classification_report(y_all, np.argmax(predictions, axis=1), target_names=EMOTIONS))

    model.save(model_save_path)
    return model

if __name__ == "__main__":
    ZIP_NAME = '/content/archive(20)-20260214T120908Z-1-001.zip'
    if os.path.exists(ZIP_NAME):
        path = extract_dataset(ZIP_NAME)
        train_emotion_model(path)
    else:
        print("Zip file not found.")

Extracting /content/archive(20)-20260214T120908Z-1-001.zip...
Loading datasets (Crema, Ravdess, Savee, Tess)...
Total samples across all datasets: 11762
Epoch 1/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.2914 - loss: 2.4026 - val_accuracy: 0.2646 - val_loss: 3.0802 - learning_rate: 1.0000e-04
Epoch 2/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.4379 - loss: 1.6483 - val_accuracy: 0.3320 - val_loss: 2.2828 - learning_rate: 1.0000e-04
Epoch 3/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.4576 - loss: 1.4535 - val_accuracy: 0.4102 - val_loss: 1.9665 - learning_rate: 1.0000e-04
Epoch 4/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.4904 - loss: 1.3068 - val_accuracy: 0.3626 - val_loss: 2.5370 - learning_rate: 1.0000e-04
Epoch 5/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5021 - loss: 1.2713 - val_accuracy: 0.4028 - val_loss: 2.1740 - learning_rate: 1.0000e-04
Epoch 6/100
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step -

In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
import librosa
import os
import zipfile
import time

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

ZIP_PATH          = '/content/archive(20)-20260214T120908Z-1-001.zip'
RAVDESS_PATH      = 'aggregated_data'
EMOTION_MODEL     = '/content/emotion_model.keras'
SAVE_PATH         = 'emotion_tts_gan.keras'

EPOCHS            = 30
BATCH_SIZE        = 16
LEARNING_RATE     = 0.0002
MAX_FILES         = None

SAMPLE_RATE       = 22050
MEL_CHANNELS      = 80
HOP_LENGTH        = 256
WIN_LENGTH        = 1024

EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

print("Config set")

def extract_dataset(zip_path, extract_to='aggregated_data'):
    if not os.path.exists(zip_path):
        print(f"Zip file not found: {zip_path}")
        return None

    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f" '{extract_to}' already exists and is not empty, skipping extraction.")
        return extract_to

    print(f"Extracting {zip_path} → {extract_to} ...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f" Extraction complete: {extract_to}")
    return extract_to

DATA_PATH = extract_dataset(ZIP_PATH, RAVDESS_PATH)
if DATA_PATH is None:
    raise FileNotFoundError(f"Could not find or extract dataset from {ZIP_PATH}")

def get_emotion_embeddings(model_path):
    model = keras.models.load_model(model_path)

    dummy = np.zeros((1, 40, 100, 3), dtype=np.float32)
    _ = model(dummy, training=False)

    output_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Dense) and layer.units == 7:
            output_layer = layer
            break

    weights = output_layer.get_weights()[0]
    embeddings = weights.T

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / (norms + 1e-8)

    print(f"Emotion embeddings: {embeddings.shape}")
    return embeddings

emotion_embeddings = get_emotion_embeddings(EMOTION_MODEL)
EMBED_DIM = emotion_embeddings.shape[1]
print(f" Embedding dim: {EMBED_DIM}")

def get_label_from_file(file_name, root_path):
    fn   = file_name.lower()
    path = root_path.lower()

    if 'tess' in path or 'oaf_' in fn or 'yaf_' in fn:
        if 'neutral'  in fn: return 0
        if 'happy'    in fn: return 1
        if 'sad'      in fn: return 2
        if 'angry'    in fn: return 3
        if 'fear'     in fn: return 4
        if 'disgust'  in fn: return 5
        if 'pleasant_surprised' in fn or 'surprise' in fn: return 6

    if fn.startswith('03-') or '-' in fn:
        parts = fn.split('-')
        if len(parts) >= 3:
            try:
                num = int(parts[2])
                if num <= 2: return 0
                return num - 2
            except ValueError:
                pass

    if '_ang_' in fn: return 3
    if '_hap_' in fn: return 1
    if '_sad_' in fn: return 2
    if '_fea_' in fn: return 4
    if '_dis_' in fn: return 5
    if '_neu_' in fn: return 0

    if 'savee' in path or fn.startswith('dc_') or fn.startswith('jk_'):
        code    = fn.split('_')[1][0] if '_' in fn else fn[0]
        mapping = {'n': 0, 'h': 1, 's': 2, 'a': 3, 'f': 4, 'd': 5, 'u': 6}
        return mapping.get(code, None)

    return None

print("Multi-dataset label function ready (RAVDESS, TESS, CREMA-D, SAVEE)")

def audio_to_mel(audio, sr):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=MEL_CHANNELS,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmax=8000
    )
    mel_db   = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_norm

def load_dataset(data_path, max_files=None):
    mels, emotions, embeddings_list = [], [], []
    count      = 0
    skipped    = 0

    for root, _, files in os.walk(data_path):
        for fname in files:
            if not fname.endswith('.wav'):
                continue

            emotion_idx = get_label_from_file(fname, root)
            if emotion_idx is None:
                skipped += 1
                continue

            try:
                fpath = os.path.join(root, fname)
                audio, sr = librosa.load(fpath, sr=SAMPLE_RATE)

                audio, _  = librosa.effects.trim(audio)
                audio     = librosa.util.fix_length(audio, size=SAMPLE_RATE * 3)

                mel = audio_to_mel(audio, SAMPLE_RATE)
                T = 100
                if mel.shape[1] < T:
                    mel = np.pad(mel, ((0, 0), (0, T - mel.shape[1])))
                else:
                    mel = mel[:, :T]

                mels.append(mel.astype(np.float32))
                emotions.append(emotion_idx)
                embeddings_list.append(
                    emotion_embeddings[emotion_idx].astype(np.float32)
                )

                count += 1
                if count % 200 == 0:
                    print(f"  Loaded {count} files...")
                if max_files and count >= max_files:
                    break

            except Exception:
                skipped += 1
                continue

        if max_files and count >= max_files:
            break

    print(f"\nTotal loaded : {count} files")
    print(f"Skipped      : {skipped} files (unrecognized format)")
    print(f"\n  Per emotion:")
    for i, name in enumerate(EMOTIONS):
        c = emotions.count(i)
        if c > 0:
            print(f"    {name:10s}: {c}")

    return (np.array(mels),
            np.array(emotions),
            np.array(embeddings_list))

print("Loading all datasets...")
mels, emotion_labels, embed_data = load_dataset(DATA_PATH, MAX_FILES)
print(f"\nDataset ready: mels={mels.shape}, embeddings={embed_data.shape}")

def build_conditioner(embed_dim):
    mel_input   = keras.Input(shape=(MEL_CHANNELS, None), name='mel')
    embed_input = keras.Input(shape=(embed_dim,),          name='embed')

    params = keras.layers.Dense(MEL_CHANNELS * 2, activation='tanh')(embed_input)
    scale  = keras.ops.expand_dims(params[:, :MEL_CHANNELS], axis=-1)
    bias   = keras.ops.expand_dims(params[:,  MEL_CHANNELS:], axis=-1)

    out = mel_input * (1.0 + scale) + bias
    return keras.Model(inputs=[mel_input, embed_input], outputs=out,
                       name='Conditioner')

def build_vocoder():
    mel_input = keras.Input(shape=(MEL_CHANNELS, None), name='mel')
    x = keras.layers.Lambda(lambda t: keras.ops.transpose(t, [0,2,1]))(mel_input)
    x = keras.layers.Conv1D(128, 7, padding='same')(x)

    ch = 128
    for stride, ksize in [(8,16),(8,16),(2,4),(2,4)]:
        x  = keras.layers.LeakyReLU(0.1)(x)
        x  = keras.layers.Conv1DTranspose(ch//2, ksize, strides=stride, padding='same')(x)
        ch //= 2
        branches = []
        for kernel in [3, 7, 11]:
            r = x
            for dil in [1, 3, 5]:
                r = keras.layers.LeakyReLU(0.1)(r)
                r = keras.layers.Conv1D(ch, kernel, padding='causal', dilation_rate=dil)(r)
                r = r + x
            branches.append(r)
        x = keras.layers.Average()(branches)

    x = keras.layers.LeakyReLU(0.1)(x)
    x = keras.layers.Conv1D(1, 7, padding='same', activation='tanh')(x)
    x = keras.layers.Lambda(lambda t: keras.ops.squeeze(t, axis=-1))(x)
    return keras.Model(inputs=mel_input, outputs=x, name='HiFiGAN')

conditioner = build_conditioner(EMBED_DIM)
vocoder     = build_vocoder()

conditioner.summary()
vocoder.summary()

optimizer = keras.optimizers.Adam(LEARNING_RATE)

@tf.function
def train_step(mel_batch, embed_batch):
    with tf.GradientTape() as tape:
        conditioned = conditioner([mel_batch, embed_batch], training=True)

        waveform = vocoder(conditioned, training=True)

        waveform_3d = tf.expand_dims(waveform, axis=-1)  # (B, T, 1)
        waveform_t  = tf.transpose(waveform_3d, [0,2,1]) # (B, 1, T)

        emotion_loss = tf.reduce_mean(tf.abs(conditioned - mel_batch))

        expected_len = mel_batch.shape[2] * HOP_LENGTH if mel_batch.shape[2] else 100 * HOP_LENGTH
        actual_len   = tf.shape(waveform)[1]
        shape_loss   = tf.cast(tf.abs(actual_len - expected_len), tf.float32) / float(expected_len)

        total_loss = emotion_loss + 0.1 * shape_loss

    trainable_vars = conditioner.trainable_variables + vocoder.trainable_variables
    grads = tape.gradient(total_loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    return total_loss, emotion_loss

def train_tts_gan():
    print("="*60)
    print("Training TTS GAN")
    print("="*60)
    print(f"Samples: {len(mels)} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}\n")

    n = len(mels)
    steps_per_epoch = n // BATCH_SIZE

    for epoch in range(EPOCHS):
        idx = np.random.permutation(n)
        mels_s   = mels[idx]
        embeds_s = embed_data[idx]

        epoch_loss = 0.0
        t0 = time.time()

        for step in range(steps_per_epoch):
            s = step * BATCH_SIZE
            e = s + BATCH_SIZE

            mel_batch   = tf.constant(mels_s[s:e],   dtype=tf.float32)
            embed_batch = tf.constant(embeds_s[s:e],  dtype=tf.float32)

            loss, em_loss = train_step(mel_batch, embed_batch)
            epoch_loss   += float(loss)

        avg_loss = epoch_loss / steps_per_epoch
        elapsed  = time.time() - t0
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s")

        if (epoch + 1) % 10 == 0:
            conditioner.save_weights(SAVE_PATH + '_conditioner')
            vocoder.save_weights(SAVE_PATH + '_vocoder')
            print(f"Checkpoint saved at epoch {epoch+1}")

    conditioner.save_weights(SAVE_PATH + '_conditioner')
    vocoder.save_weights(SAVE_PATH + '_vocoder')
    print(f"\nTraining complete!")
    print(f"Conditioner saved: {SAVE_PATH}_conditioner")
    print(f"Vocoder saved:     {SAVE_PATH}_vocoder")
    print(f"\nNow run emotion_tts.py — GAN conditioning will be active!")

train_tts_gan()

TensorFlow: 2.19.0
GPU available: True
✓ Config set
Extracting /content/archive(20)-20260214T120908Z-1-001.zip → aggregated_data ...
✓ Extraction complete: aggregated_data
Emotion embeddings: (7, 512)
 Embedding dim: 512
Multi-dataset label function ready (RAVDESS, TESS, CREMA-D, SAVEE)
Loading all datasets...
  Loaded 200 files...
  Loaded 400 files...
  Loaded 600 files...
  Loaded 800 files...
  Loaded 1000 files...
  Loaded 1200 files...
  Loaded 1400 files...
  Loaded 1600 files...
  Loaded 1800 files...
  Loaded 2000 files...
  Loaded 2200 files...
  Loaded 2400 files...
  Loaded 2600 files...
  Loaded 2800 files...
  Loaded 3000 files...
  Loaded 3200 files...
  Loaded 3400 files...
  Loaded 3600 files...
  Loaded 3800 files...
  Loaded 4000 files...
  Loaded 4200 files...
  Loaded 4400 files...
  Loaded 4600 files...
  Loaded 4800 files...
  Loaded 5000 files...
  Loaded 5200 files...
  Loaded 5400 files...
  Loaded 5600 files...
  Loaded 5800 files...
  Loaded 6000 files...
  

Model: "Conditioner"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ embed (InputLayer)  │ (None, 512)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 160)       │     82,080 │ embed[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 80)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, 80, 1)     │          0 │ get_item[0][0]    │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel (InputLayer)    │ (None, 80, None)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 80, 1)     │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 80)        │          0 │ dense[0][0]       │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 80, None)  │          0 │ mel[0][0],        │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_1       │ (None, 80, 1)     │          0 │ get_item_1[0][0]  │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 80, None)  │          0 │ multiply[0][0],   │
│                     │                   │            │ expand_dims_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,080 (320.62 KB)

 Trainable params: 82,080 (320.62 KB)

...

 Non-trainable params: 0 (0.00 B)

Training TTS GAN
Samples: 11762 | Epochs: 30 | Batch: 16



/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:855: UserWarning: Gradients do not exist for variables ['conv1d/kernel', 'conv1d/bias', 'conv1d_transpose/kernel', 'conv1d_transpose/bias', 'conv1d_1/kernel', 'conv1d_1/bias', 'conv1d_4/kernel', 'conv1d_4/bias', 'conv1d_7/kernel', 'conv1d_7/bias', 'conv1d_2/kernel', 'conv1d_2/bias', 'conv1d_5/kernel', 'conv1d_5/bias', 'conv1d_8/kernel', 'conv1d_8/bias', 'conv1d_3/kernel', 'conv1d_3/bias', 'conv1d_6/kernel', 'conv1d_6/bias', 'conv1d_9/kernel', 'conv1d_9/bias', 'conv1d_transpose_1/kernel', 'conv1d_transpose_1/bias', 'conv1d_10/kernel', 'conv1d_10/bias', 'conv1d_13/kernel', 'conv1d_13/bias', 'conv1d_16/kernel', 'conv1d_16/bias', 'conv1d_11/kernel', 'conv1d_11/bias', 'conv1d_14/kernel', 'conv1d_14/bias', 'conv1d_17/kernel', 'conv1d_17/bias', 'conv1d_12/kernel', 'conv1d_12/bias', 'conv1d_15/kernel', 'conv1d_15/bias', 'conv1d_18/kernel', 'conv1d_18/bias', 'conv1d_transpose_2/kernel', 'conv1d_transpose_2/bias', 'co

Epoch   1/30 | Loss: 0.0028 | Time: 6.9s
Epoch   2/30 | Loss: 0.0006 | Time: 2.1s
Epoch   3/30 | Loss: 0.0006 | Time: 1.6s
Epoch   4/30 | Loss: 0.0006 | Time: 1.7s
Epoch   5/30 | Loss: 0.0006 | Time: 1.6s
Epoch   6/30 | Loss: 0.0006 | Time: 1.6s
Epoch   7/30 | Loss: 0.0006 | Time: 1.6s
Epoch   8/30 | Loss: 0.0006 | Time: 1.8s
Epoch   9/30 | Loss: 0.0006 | Time: 1.9s
Epoch  10/30 | Loss: 0.0006 | Time: 1.6s


ValueError: The filename must end in `.weights.h5`. Received: filepath=emotion_tts_gan.keras_conditioner

In [ ]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
import librosa
import os
import time

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

RAVDESS_PATH      = '/content/aggregated_data'
EMOTION_MODEL     = '/content/emotion_model.keras'
SAVE_PATH         = 'emotion_tts_gan'

EPOCHS            = 30
BATCH_SIZE        = 16
LEARNING_RATE     = 0.0002
MAX_FILES         = None


SAMPLE_RATE       = 22050
MEL_CHANNELS      = 80
HOP_LENGTH        = 256
WIN_LENGTH        = 1024

EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

print(" Config set")


if not os.path.exists(RAVDESS_PATH):
    raise FileNotFoundError(f"Folder '{RAVDESS_PATH}' not found! "
                            f"Make sure your files are extracted there.")

DATA_PATH = RAVDESS_PATH
print(f" Using dataset folder: {os.path.abspath(DATA_PATH)}")


def get_emotion_embeddings(model_path):
    model = keras.models.load_model(model_path)

    dummy = np.zeros((1, 40, 100, 3), dtype=np.float32)
    _ = model(dummy, training=False)

    output_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Dense) and layer.units == 7:
            output_layer = layer
            break

    weights = output_layer.get_weights()[0]
    embeddings = weights.T

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / (norms + 1e-8)

    print(f" Emotion embeddings: {embeddings.shape}")
    return embeddings

emotion_embeddings = get_emotion_embeddings(EMOTION_MODEL)
EMBED_DIM = emotion_embeddings.shape[1]
print(f" Embedding dim: {EMBED_DIM}")


def get_label_from_file(file_name, root_path):
    fn   = file_name.lower()
    path = root_path.lower()

    if 'tess' in path or 'oaf_' in fn or 'yaf_' in fn:
        if 'neutral'  in fn: return 0
        if 'happy'    in fn: return 1
        if 'sad'      in fn: return 2
        if 'angry'    in fn: return 3
        if 'fear'     in fn: return 4
        if 'disgust'  in fn: return 5
        if 'pleasant_surprised' in fn or 'surprise' in fn: return 6

    if fn.startswith('03-') or '-' in fn:
        parts = fn.split('-')
        if len(parts) >= 3:
            try:
                num = int(parts[2])
                if num <= 2: return 0
                return num - 2
            except ValueError:
                pass

    if '_ang_' in fn: return 3
    if '_hap_' in fn: return 1
    if '_sad_' in fn: return 2
    if '_fea_' in fn: return 4
    if '_dis_' in fn: return 5
    if '_neu_' in fn: return 0

    if 'savee' in path or fn.startswith('dc_') or fn.startswith('jk_'):
        code    = fn.split('_')[1][0] if '_' in fn else fn[0]
        mapping = {'n': 0, 'h': 1, 's': 2, 'a': 3, 'f': 4, 'd': 5, 'u': 6}
        return mapping.get(code, None)

    return None

print(" Multi-dataset label function ready (RAVDESS, TESS, CREMA-D, SAVEE)")

def audio_to_mel(audio, sr):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=MEL_CHANNELS,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        fmax=8000
    )
    mel_db   = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_norm  # shape: (80, T)

def load_dataset(data_path, max_files=None):
    mels, emotions, embeddings_list = [], [], []
    count      = 0
    skipped    = 0

    for root, _, files in os.walk(data_path):
        for fname in files:
            if not fname.endswith('.wav'):
                continue

            emotion_idx = get_label_from_file(fname, root)
            if emotion_idx is None:
                skipped += 1
                continue

            try:
                fpath = os.path.join(root, fname)
                audio, sr = librosa.load(fpath, sr=SAMPLE_RATE)

                audio, _  = librosa.effects.trim(audio)
                audio     = librosa.util.fix_length(audio, size=SAMPLE_RATE * 3)

                mel = audio_to_mel(audio, SAMPLE_RATE)

                T = 100
                if mel.shape[1] < T:
                    mel = np.pad(mel, ((0, 0), (0, T - mel.shape[1])))
                else:
                    mel = mel[:, :T]

                mels.append(mel.astype(np.float32))
                emotions.append(emotion_idx)
                embeddings_list.append(
                    emotion_embeddings[emotion_idx].astype(np.float32)
                )

                count += 1
                if count % 200 == 0:
                    print(f"  Loaded {count} files...")
                if max_files and count >= max_files:
                    break

            except Exception:
                skipped += 1
                continue

        if max_files and count >= max_files:
            break

    print(f"\nTotal loaded : {count} files")
    print(f"  Skipped      : {skipped} files (unrecognized format)")
    print(f"\n  Per emotion:")
    for i, name in enumerate(EMOTIONS):
        c = emotions.count(i)
        if c > 0:
            print(f"    {name:10s}: {c}")

    return (np.array(mels),
            np.array(emotions),
            np.array(embeddings_list))

print("Loading all datasets...")
mels, emotion_labels, embed_data = load_dataset(DATA_PATH, MAX_FILES)
print(f"\n✓ Dataset ready: mels={mels.shape}, embeddings={embed_data.shape}")

def build_conditioner(embed_dim):
    mel_input   = keras.Input(shape=(MEL_CHANNELS, None), name='mel')
    embed_input = keras.Input(shape=(embed_dim,),          name='embed')

    params = keras.layers.Dense(MEL_CHANNELS * 2, activation='tanh')(embed_input)
    scale  = keras.ops.expand_dims(params[:, :MEL_CHANNELS], axis=-1)
    bias   = keras.ops.expand_dims(params[:,  MEL_CHANNELS:], axis=-1)

    out = mel_input * (1.0 + scale) + bias
    return keras.Model(inputs=[mel_input, embed_input], outputs=out,
                       name='Conditioner')

def build_vocoder():
    mel_input = keras.Input(shape=(MEL_CHANNELS, None), name='mel')
    x = keras.layers.Lambda(lambda t: keras.ops.transpose(t, [0,2,1]))(mel_input)
    x = keras.layers.Conv1D(128, 7, padding='same')(x)

    ch = 128
    for stride, ksize in [(8,16),(8,16),(2,4),(2,4)]:
        x  = keras.layers.LeakyReLU(0.1)(x)
        x  = keras.layers.Conv1DTranspose(ch//2, ksize, strides=stride, padding='same')(x)
        ch //= 2
        branches = []
        for kernel in [3, 7, 11]:
            r = x
            for dil in [1, 3, 5]:
                r = keras.layers.LeakyReLU(0.1)(r)
                r = keras.layers.Conv1D(ch, kernel, padding='causal', dilation_rate=dil)(r)
                r = r + x
            branches.append(r)
        x = keras.layers.Average()(branches)

    x = keras.layers.LeakyReLU(0.1)(x)
    x = keras.layers.Conv1D(1, 7, padding='same', activation='tanh')(x)
    x = keras.layers.Lambda(lambda t: keras.ops.squeeze(t, axis=-1))(x)
    return keras.Model(inputs=mel_input, outputs=x, name='HiFiGAN')

conditioner = build_conditioner(EMBED_DIM)
vocoder     = build_vocoder()

conditioner.summary()
vocoder.summary()

optimizer = keras.optimizers.Adam(LEARNING_RATE)

@tf.function
def train_step(mel_batch, embed_batch):
    with tf.GradientTape() as tape:

        conditioned_mel = conditioner([mel_batch, embed_batch], training=True)

        waveform = vocoder(conditioned_mel, training=True)

        waveform_2d = tf.expand_dims(waveform, axis=-1)

        target_len  = tf.shape(mel_batch)[2]
        waveform_ds = tf.image.resize(
            tf.expand_dims(waveform_2d, 1),
            [1, target_len]
        )
        waveform_ds = tf.squeeze(waveform_ds, axis=[1, 3])
        waveform_ds = tf.expand_dims(waveform_ds, axis=1)

        waveform_mel = tf.tile(waveform_ds, [1, MEL_CHANNELS, 1])

        conditioner_loss = tf.reduce_mean(tf.abs(conditioned_mel - mel_batch))

        vocoder_loss = tf.reduce_mean(tf.abs(waveform_mel - conditioned_mel))

        total_loss = conditioner_loss + vocoder_loss
    trainable_vars = (conditioner.trainable_variables +
                      vocoder.trainable_variables)
    grads = tape.gradient(total_loss, trainable_vars)

    valid = [(g, v) for g, v in zip(grads, trainable_vars) if g is not None]
    if valid:
        optimizer.apply_gradients(valid)

    return total_loss, conditioner_loss, vocoder_loss

def train_tts_gan():
    print("="*60)
    print("Training TTS GAN")
    print("="*60)
    print(f"Samples: {len(mels)} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}\n")

    n = len(mels)
    steps_per_epoch = n // BATCH_SIZE

    for epoch in range(EPOCHS):
        idx = np.random.permutation(n)
        mels_s   = mels[idx]
        embeds_s = embed_data[idx]

        epoch_loss = 0.0
        t0 = time.time()

        for step in range(steps_per_epoch):
            s = step * BATCH_SIZE
            e = s + BATCH_SIZE

            mel_batch   = tf.constant(mels_s[s:e],   dtype=tf.float32)
            embed_batch = tf.constant(embeds_s[s:e],  dtype=tf.float32)

            loss, c_loss, v_loss = train_step(mel_batch, embed_batch)
            epoch_loss          += float(loss)

        avg_loss = epoch_loss / steps_per_epoch
        elapsed  = time.time() - t0
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} "
              f"(cond: {float(c_loss):.4f}, voc: {float(v_loss):.4f}) | Time: {elapsed:.1f}s")

        if (epoch + 1) % 10 == 0:
            conditioner.save_weights(SAVE_PATH + '_conditioner.weights.h5')
            vocoder.save_weights(SAVE_PATH + '_vocoder.weights.h5')
            print(f"Checkpoint saved at epoch {epoch+1}")

    conditioner.save_weights(SAVE_PATH + '_conditioner.weights.h5')
    vocoder.save_weights(SAVE_PATH + '_vocoder.weights.h5')
    print(f"\n Training complete!")
    print(f" Conditioner saved: {SAVE_PATH}_conditioner.weights.h5")
    print(f" Vocoder saved:     {SAVE_PATH}_vocoder.weights.h5")
    print(f"\nNow run emotion_tts.py — GAN conditioning will be active!")

train_tts_gan()

TensorFlow: 2.19.0
GPU available: True
✓ Config set
✓ Using dataset folder: /content/aggregated_data
✓ Emotion embeddings: (7, 512)
✓ Embedding dim: 512
✓ Multi-dataset label function ready (RAVDESS, TESS, CREMA-D, SAVEE)
Loading all datasets...
  Loaded 200 files...
  Loaded 400 files...
  Loaded 600 files...
  Loaded 800 files...
  Loaded 1000 files...
  Loaded 1200 files...
  Loaded 1400 files...
  Loaded 1600 files...
  Loaded 1800 files...
  Loaded 2000 files...
  Loaded 2200 files...
  Loaded 2400 files...
  Loaded 2600 files...
  Loaded 2800 files...
  Loaded 3000 files...
  Loaded 3200 files...
  Loaded 3400 files...
  Loaded 3600 files...
  Loaded 3800 files...
  Loaded 4000 files...
  Loaded 4200 files...
  Loaded 4400 files...
  Loaded 4600 files...
  Loaded 4800 files...
  Loaded 5000 files...
  Loaded 5200 files...
  Loaded 5400 files...
  Loaded 5600 files...
  Loaded 5800 files...
  Loaded 6000 files...
  Loaded 6200 files...
  Loaded 6400 files...
  Loaded 6600 files...

Model: "Conditioner"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ embed (InputLayer)  │ (None, 512)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 160)       │     82,080 │ embed[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 80)        │          0 │ dense_1[0][0]     │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_2       │ (None, 80, 1)     │          0 │ get_item_2[0][0]  │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel (InputLayer)    │ (None, 80, None)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_38 (Add)        │ (None, 80, 1)     │          0 │ expand_dims_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 80)        │          0 │ dense_1[0][0]     │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 80, None)  │          0 │ mel[0][0],        │
│ (Multiply)          │                   │            │ add_38[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_3       │ (None, 80, 1)     │          0 │ get_item_3[0][0]  │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_39 (Add)        │ (None, 80, None)  │          0 │ multiply_1[0][0], │
│                     │                   │            │ expand_dims_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 82,080 (320.62 KB)

 Trainable params: 82,080 (320.62 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel (InputLayer)    │ (None, 80, None)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, None, 80)  │          0 │ mel[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_38 (Conv1D)  │ (None, None, 128) │     71,808 │ lambda_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_41      │ (None, None, 128) │          0 │ conv1d_38[0][0]   │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_transpose_4  │ (None, None, 64)  │    131,136 │ leaky_re_lu_41[0… │
│ (Conv1DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_42      │ (None, None, 64)  │          0 │ conv1d_transpose… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_45      │ (None, None, 64)  │          0 │ conv1d_transpose… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_48      │ (None, None, 64)  │          0 │ conv1d_transpose… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_39 (Conv1D)  │ (None, None, 64)  │     12,352 │ leaky_re_lu_42[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_42 (Conv1D)  │ (None, None, 64)  │     28,736 │ leaky_re_lu_45[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_45 (Conv1D)  │ (None, None, 64)  │     45,120 │ leaky_re_lu_48[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_40 (Add)        │ (None, None, 64)  │          0 │ conv1d_39[0][0],  │
│                     │                   │            │ conv1d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_43 (Add)        │ (None, None, 64)  │          0 │ conv1d_42[0][0],  │
│                     │                   │            │ conv1d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_46 (Add)        │ (None, None, 64)  │          0 │ conv1d_45[0][0],  │
│                     │                   │            │ conv1d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_43      │ (None, None, 64)  │          0 │ add_40[0][0]      │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_46      │ (None, None, 64)  │          0 │ add_43[0][0]      │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_49      │ (None, None, 64)  │          0 │ add_46[0][0]      │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_40 (Conv1D)  │ (None, None, 64)  │     12,352 │ leaky_re_lu_43[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_43 (Conv1D)  │ (None, None, 64)  │     28,736 │ leaky_re_lu_46[0… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 582,185 (2.22 MB)

 Trainable params: 582,185 (2.22 MB)

 Non-trainable params: 0 (0.00 B)

Training TTS GAN
Samples: 11762 | Epochs: 30 | Batch: 16

Epoch   1/30 | Loss: 0.5247 (cond: 0.0020, voc: 0.4866) | Time: 132.2s
Epoch   2/30 | Loss: 0.5165 (cond: 0.0021, voc: 0.4773) | Time: 114.7s
Epoch   3/30 | Loss: 0.5156 (cond: 0.0022, voc: 0.5372) | Time: 115.3s
Epoch   4/30 | Loss: 0.5151 (cond: 0.0022, voc: 0.5008) | Time: 115.0s
Epoch   5/30 | Loss: 0.5147 (cond: 0.0019, voc: 0.5690) | Time: 115.2s
Epoch   6/30 | Loss: 0.5146 (cond: 0.0018, voc: 0.5202) | Time: 114.8s
Epoch   7/30 | Loss: 0.5146 (cond: 0.0024, voc: 0.5024) | Time: 114.7s
Epoch   8/30 | Loss: 0.5144 (cond: 0.0023, voc: 0.5505) | Time: 115.0s
Epoch   9/30 | Loss: 0.5144 (cond: 0.0023, voc: 0.5642) | Time: 114.9s
Epoch  10/30 | Loss: 0.5142 (cond: 0.0024, voc: 0.5338) | Time: 114.8s
  ✓ Checkpoint saved at epoch 10
Epoch  11/30 | Loss: 0.5142 (cond: 0.0022, voc: 0.5144) | Time: 114.8s
Epoch  12/30 | Loss: 0.5141 (cond: 0.0026, voc: 0.5224) | Time: 114.4s
Epoch  13/30 | Loss: 0.5141 (cond: 0.0022, voc: 0.5341) |